In [12]:
import os,sys

# Set the path to the parent directory manually
parent_dir = os.path.abspath("../..")
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
    
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
import pkg_resources
from datetime import datetime as dt, timedelta
from datetime import datetime
import glob
from netCDF4 import Dataset
from util.wrf_process import (calc_derive, object_tracking, read_and_write, to_polar)
from util.ml_framework import (cnn, vae)
from wrf import (to_np, getvar, smooth2d, get_cartopy, cartopy_xlim,
                 cartopy_ylim, latlon_coords, interplevel, ll_to_xy)
import gc,pickle
from tqdm import tqdm
import xarray as xr
from natsort import natsorted
from util.ml_preprocess import data_preproc
#from vae3d import VAEEncoder, VAEDecoder, VAE, elbo_loss, weights_init
import optuna
import random,math
import proplot as plot

In [25]:
def get_shape(x):
    try:
        return np.asarray(x).shape
    except Exception:
        # fallback: manual inspection
        try:
            n0 = len(x)
            if n0 == 0:
                return (0,)
            n1 = len(x[0])
            if n1 == 0:
                return (n0, 0)
            n2 = len(x[0][0])
            return (n0, n1, n2)
        except Exception:
            return None  # completely broken

In [32]:
class calc_diag:
    def __init__(self,WRFFILELIST,trackdata,startindx,preslv,zlv,weshape,nsshape):
        self.WRFFILELIST = WRFFILELIST
        self.trackdata = trackdata
        self.startindx = startindx
        self.preslv = preslv
        self.zlv = zlv
        self.weshape = weshape
        self.nsshape = nsshape

    def interp_to_preslv(self,vartointerp,prestointerp):
        nc_wp = [interplevel(vartointerp,prestointerp,i) for i in self.preslv]
        return np.asarray(nc_wp)

    def interp_to_zlv(self,vartointerp,prestointerp):
        nc_wp = [interplevel(vartointerp,prestointerp,i) for i in self.zlv]
        return np.asarray(nc_wp)
        
    def read_vars(self,varname):
        varout = []
        for i in range(len(self.WRFFILELIST)):
            varout.append(getvar(Dataset(self.WRFFILELIST[i]),varname))
        return np.asarray(varout)

    def read_vars_and_interp(self,varname,TYPE):
        varout = []
        for i in range(len(self.WRFFILELIST)):
            vartointerp = getvar(Dataset(self.WRFFILELIST[i]),varname)
            if TYPE=='pres':
                prestointerp = getvar(Dataset(self.WRFFILELIST[i]),'pres')
                varout.append(self.interp_to_preslv(vartointerp,prestointerp))
            elif TYPE=='height_agl':
                ztointerp = getvar(Dataset(self.WRFFILELIST[i]),'height_agl')
                varout.append(self.interp_to_zlv(vartointerp,ztointerp))
        return np.asarray(varout)

    def read_single_lv_vars(self,varname,TYPE):
        varout = []
        for i in range(len(self.WRFFILELIST)):
            varout.append(getvar(Dataset(self.WRFFILELIST[i]),varname))
        return np.asarray(varout)
        
    def find_track_ij(self):
        ixs,iys = [],[]
        for i, WRFFILE in tqdm(enumerate(self.WRFFILELIST)):
            # read file
            nc_ctrl = Dataset(WRFFILE)
            # Target location
            target_lat = self.trackdata['clat'][self.startindx+i] # Simulations start from 12 UTC
            target_lon = self.trackdata['clon'][self.startindx+i] # Simulations start from 12 UTC
            # Compute track indices
            ix, iy = ll_to_xy(nc_ctrl, target_lat, target_lon, timeidx=0) #x, y
            ixs.append(int(ix.values))
            iys.append(int(iy.values))
        return ixs,iys
        
    def compute_tc_shear(self, u, v, x0, y0, dx_km=3.0,
                     inner_km=200.0, outer_km=800.0,
                     p_top=200.0, p_bot=850.0):
        """
        Compute mean 200–850 hPa shear vector in a TC-centered annulus.

        Parameters
        ----------
        ds : netCDF Dataset (opened with netCDF4 or xarray + wrf-python)
            WRF output file handle
        x0, y0 : int
            TC center grid indices (x,y) in WRF domain
        dx_km : float
            Grid spacing (km)
        inner_km, outer_km : float
            Inner and outer radius of annulus (km)
        p_top, p_bot : float
            Top and bottom pressure levels (hPa)

        Returns
        -------
        shear_u, shear_v : floats
            Zonal and meridional shear components (m/s)
        shear_mag : float
            Shear magnitude (m/s)
        """

        # --- Interpolate to levels
        u200 = u[self.preslv.index(p_top),...]
        v200 = v[self.preslv.index(p_top),...]
        u850 = u[self.preslv.index(p_bot),...]
        v850 = v[self.preslv.index(p_bot),...]

        # --- Make distance mask in km
        ny, nx = u200.shape
        X, Y = np.meshgrid(np.arange(nx), np.arange(ny))
        dx = (X - x0) * dx_km
        dy = (Y - y0) * dx_km
        r = np.sqrt(dx**2 + dy**2)

        mask = (r >= inner_km) & (r <= outer_km)

        # --- Area-average wind in annulus
        u200m = np.nanmean(np.where(mask, u200, np.nan))
        v200m = np.nanmean(np.where(mask, v200, np.nan))
        u850m = np.nanmean(np.where(mask, u850, np.nan))
        v850m = np.nanmean(np.where(mask, v850, np.nan))

        # --- Shear vector
        shear_u = u200m - u850m
        shear_v = v200m - v850m
        shear_mag = np.sqrt(shear_u**2 + shear_v**2)

        return shear_u, shear_v, shear_mag

    def get_shear_mag_dir(self,Upres,Vpres,Xs,ys,settings=[200.0,800.0,200.0,850.0,3.0]):
        """
        settings = [inner_km,outer_km,p_top,p_bot,dx_km]
        """
        shear_mags,shear_dirs = [],[]
        shear_us, shear_vs = [],[]
        for i in range(len(Xs)):
            radiusshape = int(settings[1]/settings[4])
            if ((self.weshape-Xs[i])>radiusshape) and ((self.nsshape-ys[i])>radiusshape):
                shear_mags.append(np.nan)
                shear_dirs.append(np.nan)
                shear_us.append(np.nan)
                shear_vs.append(np.nan)
            else:
                shear_u, shear_v, shear_mag = self.compute_tc_shear(Upres[i,...], Vpres[i,...], Xs[i], ys[i], 
                                                                       dx_km=settings[4],
                                                                       inner_km=settings[0], outer_km=settings[1],
                                                                       p_top=settings[2], p_bot=settings[3])
                shear_mags.append(shear_mag)
                shear_dirs.append((np.degrees(np.arctan2(shear_v, shear_u)) % 360.0))
                shear_us.append(shear_u)
                shear_vs.append(shear_v)
        return shear_mags,shear_dirs, shear_vs, shear_us

    def data_to_TCcentre(self,data,Xs,ys,settings=[800.0,3.0,200.0]):
        dataTC = []
        for i in range(len(Xs)):
            radiusshape = int(settings[0]/settings[1])
            if ((self.weshape-Xs[i])>radiusshape) and ((self.nsshape-ys[i])>radiusshape):
                dataTC.append(np.nan)
            else:
                domainint = int(settings[2])
                if len(data.shape)==4:
                    dataTC.append(data[i,:,ys[i]-domainint:ys[i]+domainint,Xs[i]-domainint:Xs[i]+domainint])
                    print(data[i,:,ys[i]-domainint:ys[i]+domainint,Xs[i]-domainint:Xs[i]+domainint].shape)
                elif len(data.shape)==3:
                    dataTC.append(data[i,ys[i]-domainint:ys[i]+domainint,Xs[i]-domainint:Xs[i]+domainint])
        # combined version
        shapes = [get_shape(x) for x in dataTC]
        good_shape = (11, 400, 400)

        bad_indices = [i for i, s in enumerate(shapes) if s != good_shape]

        print("bad_indices =", bad_indices)

        # remove
        dataTC = [x for i, x in enumerate(dataTC) if i not in bad_indices]

        # now safe to convert
        dataTC = np.asarray(dataTC)
        print("final shape:", dataTC.shape)
        return np.asarray(dataTC), bad_indices
            
    def data_to_polar(self,Cnc_wpz,dx,rmax,dr,nazim):        
        # Convert to polar coordinates
        # Suppose Xtrain_n shape = (n_samples, 9, ny, nx)
        ny, nx = Cnc_wpz.shape[-2:]
        if len(Cnc_wpz.shape)>3:
            tri, pts, target_points, r, az = to_polar.precompute_cartesian_to_polar_map(
                nx=nx, ny=ny, dx=dx, rmax=rmax, dr=dr, nazim=nazim
            )
            return to_polar.fast_cartesian_to_polar_batch(Cnc_wpz, tri, pts, target_points, r, az), r, az
        elif len(Cnc_wpz.shape)==3:
            Cnc_wpza = np.expand_dims(Cnc_wpz, axis=1)
            tri, pts, target_points, r, az = to_polar.precompute_cartesian_to_polar_map(
                nx=nx, ny=ny, dx=dx, rmax=rmax, dr=dr, nazim=nazim
            )
            return to_polar.fast_cartesian_to_polar_batch(Cnc_wpza, tri, pts, target_points, r, az), r, az

In [29]:
def get_proc_data_polar(track,ctrl_files,wantvar='rh',TYPE='height_agl'):    
    etalevels,weshape,nsshape = getvar(Dataset(ctrl_files[0]),'pres').shape
    wrfdiags = calc_diag(ctrl_files,track,12,[10000,20000,30000,40000,50000,60000,70000,80000,85000,90000,100000],
                         [500,1000,1500,2000,2500,3000,3500,4000,4500,5000,5500,6000,6500,7000,7500,8000,8500,9000,9500,10000],
                         weshape,nsshape)
    tracks_ij = wrfdiags.find_track_ij()

    # Shear calculation
    Upres,Vpres = wrfdiags.read_vars_and_interp('ua','pres'), wrfdiags.read_vars_and_interp('va','pres')

    shear_mags,shear_dirs,shear_uv,shear_vs = wrfdiags.get_shear_mag_dir(Upres,Vpres,tracks_ij[0],
                                                                         tracks_ij[1],
                                                                         settings=[200.0,800.0,int(200.0*100),int(850.0*100),3.0])
    # Calculate polar version of one variable
    if TYPE=='single':
        RHpres = wrfdiags.read_single_lv_vars(wantvar,TYPE)
    else:
        RHpres = wrfdiags.read_vars_and_interp(wantvar,TYPE)
    RHpres_ctre, bad_indices = wrfdiags.data_to_TCcentre(RHpres,tracks_ij[0],tracks_ij[1])
    RHpres_plr, r, az = wrfdiags.data_to_polar(RHpres_ctre,3,600,3,360)
    return {'trackkij':tracks_ij,'shearmags':shear_mags,'sheardirs':shear_dirs,'varcart':RHpres_ctre,'varpol':RHpres_plr, 'r':r, 'az':az, 'bad_indices':bad_indices}

In [30]:
class var_processor:
    def __init__(self, pres, z, track, exps, expname):
        self.pres = pres # Pressure levels
        self.z = z # AGL Height levels
        self.track = track # TC track
        self.exps = exps # List storing experiment filenames (list)
        self.expname = expname # Name of experiments

    def proc_onevar(self,wantvar,TYPE):
        """
        wantvar: Variable name to extract and convert (wrf-python)
        TYPE: Integrate to height AGL (height_agl) or pressure (pres)
        """
        vardict = {}
        for expname,expfilelist in zip(self.expname,self.exps):
            vardict[expname] = get_proc_data_polar(self.track,expfilelist,wantvar=wantvar,TYPE=TYPE)
        return vardict

    def rotating_polar_onevar(self,vardict,sheardict):
        rotated_vardict = {}
        for expnamez in self.expname:
            if type(vardict[expnamez])==np.ndarray:
                temp = vardict[expnamez]
            else:
                temp = vardict[expnamez]['varpol']
            sheardir = sheardict[expnamez]['sheardirs']
            rotatemp = np.asarray([np.roll(temp[i],shift=-int(sheardir[i]), axis=-1) for i in range(len(temp))])
            rotated_vardict[expnamez] = rotatemp
        return rotated_vardict


In [33]:
pres = np.array([10000,20000,30000,40000,50000,60000,70000,80000,85000,90000,100000])
z = np.array([500,1000,1500,2000,2500,3000,3500,4000,4500,5000,6000,6500,7000,7500,8000,8500,9000,9500,10000])

i = '06'
track = xr.open_dataset(f'/glade/derecho/scratch/ihtam/temp/memb{i}/track_avor_850-600.nc')
ctrl_files = sorted(glob.glob(f"/glade/derecho/scratch/ihtam/temp/memb{i}/wrfout_d02_2013-11-0*"))[1:1+24]
var_init = var_processor(pres, z, track, [ctrl_files], 
                            [i])
tk_dict = var_init.proc_onevar('tk','pres')
#save_to_pickle(f'./store/tk_dict_10ensm_memb{i}.pkl',tk_dict)

24it [00:03,  6.63it/s]


(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 400, 400)
(11, 0, 400)
(11, 0, 400)
(11, 0, 400)
(11, 0, 400)
(11, 0, 400)
(11, 400, 400)
(11, 400, 400)
bad_indices = [14, 15, 16, 17, 18, 21, 22, 23]
final shape: (16, 11, 400, 400)


100%|██████████| 16/16 [00:05<00:00,  2.93it/s]
